# 06 — RQ4 re-verdict on the corrected DKVMN: junyi

Applies the **same pre-registered rule, unchanged** — comparator outside the top 3 within
session AND first at every adequate real gap horizon, on the confirmatory seed — to the
corrected `table9` files produced by `05_DKVMN_REFIT__junyi__seed*`.

The rule is not re-specified here. If the verdict changes, that is the finding.

In [1]:
# --- locate pcdt_p2.py and p2_runtime.py -----------------------------------------
import os, sys, json, time, glob
_here = os.path.abspath("")
for _c in (_here, os.path.dirname(_here), os.path.join(_here, "..")):
    if os.path.exists(os.path.join(_c, "pcdt_p2.py")):
        sys.path.insert(0, os.path.abspath(_c)); break
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg") if os.environ.get("P2_HEADLESS") else None
import matplotlib.pyplot as plt
import pcdt_p2 as P
import p2_runtime as R
pd.set_option("display.width", 220)
print("pcdt_p2   :", P.__file__)
print("p2_runtime:", R.__file__)
assert hasattr(P.Config(), "dkvmn_legacy_lag"), (
    "This notebook requires the CORRECTED pcdt_p2.py. The copy on the path is the "
    "archived one, in which DKVMN reads its value memory before writing the current "
    "interaction. Refitting against it would reproduce the defect, not repair it.")


pcdt_p2   : /home/hamed/Downloads/Research/P02/pcdt_p2.py
p2_runtime: /home/hamed/Downloads/Research/P02/p2_runtime.py


In [2]:
RUN_ID   = "RQ4FIX__junyi"
OUT_ROOT = "/home/hamed/Downloads/Research/P02/PCDT_P2_runs"
DATASET  = "junyi"
SEEDS    = {0: 20260202, 1: 20260203, 2: 20260204}
CONFIRMATORY = 2                      # pre-registered; not re-chosen here
CFG = P.Config(dataset=DATASET, out_root=OUT_ROOT, seed_index=CONFIRMATORY)
CTX = R.RunContext(RUN_ID, OUT_ROOT, cfg=CFG,
                   inputs_expected=[f"DKVMNFIX__{DATASET}__seed{i}/table9" for i in SEEDS],
                   description="RQ4 re-verdict with corrected DKVMN — junyi")
DIRS = CTX.dirs

RUN RQ4FIX__junyi
RQ4 re-verdict with corrected DKVMN — junyi
started   2026-08-19 09:49:54   pid 8912
writes to /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4FIX__junyi
reads     DKVMNFIX__junyi__seed0/table9, DKVMNFIX__junyi__seed1/table9, DKVMNFIX__junyi__seed2/table9


In [3]:
def load9(prefix, si):
    b = CTX.sibling(f"{prefix}__{DATASET}__seed{si}")
    if b is None:
        return None
    f = CTX.read_input(os.path.join(b, "tables", "table9_postgap_auc_by_model.csv"),
                       label=f"{prefix}__{DATASET}__seed{si}/table9", required=False)
    return pd.read_csv(f) if f else None

NEW = {si: load9("DKVMNFIX", si) for si in SEEDS}
OLD = {si: load9("RQ2RQ3", si) for si in SEEDS}
missing = [si for si, t in NEW.items() if t is None]
if missing:
    raise FileNotFoundError(
        f"corrected table9 missing for seed index {missing}. Run "
        f"05_DKVMN_REFIT__{DATASET}__seed<n>.ipynb for those seeds first. "
        "No verdict is issued on partial inputs.")

  read input     DKVMNFIX__junyi__seed0/table9
  read input     DKVMNFIX__junyi__seed1/table9
  read input     DKVMNFIX__junyi__seed2/table9
  read input     RQ2RQ3__junyi__seed0/table9
  read input     RQ2RQ3__junyi__seed1/table9
  read input     RQ2RQ3__junyi__seed2/table9


In [4]:
def verdict(T9):
    t3 = T9[CONFIRMATORY]
    if t3["auc_mean_postgap"].isna().all():
        raise RuntimeError(
            "NO real gap horizon met the adequacy floor on the confirmatory seed, so the "
            "comparator cannot be named and the rule cannot be evaluated. This is a "
            "refusal, not a null result: report it as 'not testable on these data'.")
    comparator = t3.loc[t3["auc_mean_postgap"].idxmax(), "model"]
    rows = []
    for si in SEEDS:
        t = T9[si]
        if t is None:
            continue
        within_rank = int(t.loc[t.model == comparator, "rank_within"].iloc[0])
        adeq = [h for h in P.REAL_HORIZONS if t[f"auc_{h}"].notna().any()]
        leaders = {h: t.loc[t[f"auc_{h}"].idxmax(), "model"] for h in adeq}
        leg1 = within_rank > 3
        leg2 = bool(adeq) and all(v == comparator for v in leaders.values())
        rows.append(dict(seed=SEEDS[si], role=("confirmatory" if si == CONFIRMATORY
                                               else "exploratory"),
                         comparator=comparator, comparator_within_rank=within_rank,
                         leg1_outside_top3=leg1,
                         gap_leaders="; ".join(f"{h}:{leaders[h]}" for h in adeq),
                         leg2_first_all_gaps=leg2, divergence=bool(leg1 and leg2)))
    return comparator, pd.DataFrame(rows), adeq

c_old, P_old, adeq_old = verdict(OLD)
c_new, P_new, adeq_new = verdict(NEW)
print("=== ARCHIVED (DKVMN reading one interaction short) ===")
print(P_old.to_string(index=False))
print("\n=== CORRECTED ===")
print(P_new.to_string(index=False))

v_old = bool(P_old.loc[P_old.role == "confirmatory", "divergence"].iloc[0])
v_new = bool(P_new.loc[P_new.role == "confirmatory", "divergence"].iloc[0])
print("\n" + "=" * 78)
print(f"RQ4 on {DATASET}:  archived {'CONFIRMED' if v_old else 'NOT CONFIRMED'} "
      f"(comparator {c_old})  ->  corrected "
      f"{'CONFIRMED' if v_new else 'NOT CONFIRMED'} (comparator {c_new})")
print("=" * 78)
if v_old != v_new:
    print("*** THE VERDICT CHANGED. The archived finding was conditional on an "
          "implementation defect in one comparator and must not be reported as it "
          "stands. ***")
elif c_old != c_new:
    print("*** The verdict is unchanged but the comparator is a different model. The "
          "claim must be restated against the new comparator. ***")
else:
    print("Verdict and comparator both unchanged: the finding survives the correction.")

=== ARCHIVED (DKVMN reading one interaction short) ===
    seed         role comparator  comparator_within_rank  leg1_outside_top3                   gap_leaders  leg2_first_all_gaps  divergence
20260202  exploratory      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False
20260203  exploratory      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False
20260204 confirmatory      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False

=== CORRECTED ===
    seed         role comparator  comparator_within_rank  leg1_outside_top3                   gap_leaders  leg2_first_all_gaps  divergence
20260202  exploratory      DKVMN                       3              False d1:DKVMN; d7:DKVMN; d28:DKVMN                 True       False
20260203  exploratory      DKVMN                       3              False d1:DKVMN; d7:DKV

In [5]:
VER = dict(dataset=DATASET, rule="UNCHANGED from the pre-registration: comparator ranks "
           "outside the top 3 within session AND first at every adequate real gap horizon, "
           "on the confirmatory seed",
           archived=dict(comparator=c_old, confirmed=v_old),
           corrected=dict(comparator=c_new, confirmed=v_new,
                          adequate_horizons=adeq_new),
           verdict_changed=bool(v_old != v_new))
P.save_tables({"table11_primary_rule_archived": P_old,
               "table11_primary_rule_corrected": P_new}, DIRS["tab"])
with open(os.path.join(DIRS["art"], "verdict_rq4_corrected.json"), "w") as fh:
    json.dump(VER, fh, indent=2, default=str)
CTX.finalize(status="complete", extra=dict(research_question="RQ4 (corrected DKVMN)",
                                           dataset=DATASET, verdict=VER))

tables -> /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4FIX__junyi/tables: ['table11_primary_rule_archived', 'table11_primary_rule_corrected']

RUN RQ4FIX__junyi — complete
  runtime        0.0 min
  files written  6
  inputs read    6 (0 missing)
  isolation      OK — no finished run was modified
  manifest       /home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4FIX__junyi/manifest.json


{'run_id': 'RQ4FIX__junyi',
 'description': 'RQ4 re-verdict with corrected DKVMN — junyi',
 'status': 'complete',
 'notes': '',
 'fixture': False,
 'started': '2026-08-19 09:49:54',
 'finished': '2026-08-19 09:49:55',
 'runtime_minutes': 0.0,
 'output_dir': '/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/RQ4FIX__junyi',
 'environment': {'python': '3.13.9',
  'platform': 'Linux-7.0.0-28-generic-x86_64-with-glibc2.39',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'torch': '2.12.0.dev20260408+cu128',
  'sklearn': '1.7.2',
  'scipy': '1.18.0',
  'matplotlib': '3.10.6',
  'cuda': True,
  'gpu': 'NVIDIA GeForce RTX 5080'},
 'config': None,
 'inputs': [{'label': 'DKVMNFIX__junyi__seed0/table9',
   'path': '/home/hamed/Downloads/Research/P02/PCDT_P2_runs/runs/DKVMNFIX__junyi__seed0/tables/table9_postgap_auc_by_model.csv',
   'present': True,
   'sha256': '45883d738668d87741fa0bf77e54c32bc353332a24bb2cbfafb664adc7bc94e5',
   'bytes': 1379,
   'mtime': '2026-08-18 20:25:54'},
  {'label': 'DK